# Unsupervised Learning

## Learning Objectives
* Being able to apply dimensionality reduction using Principal Component Analysis (PCA) to reduce the number of features in a dataset.
* Being able to apply the K-Means algorithm to find clusters in data.
* Being able to determine the optimal number of clusters using the silhouette coefficient.

## Dimensionality Reduction


In the [lecture](https://janalasser.at/lectures/MD_KI/VO3_3_data_transformations/), we learned about dimensionality reduction as a tool to overcome the [curse of dimensionality](https://janalasser.at/lectures/MD_KI/VO3_3_data_transformations/#/2/2/0). This means that our dataset has *too many* features to handle effectively. In particular, in [unsupervised learning](https://janalasser.at/lectures/MD_KI/VO3_1_unsupervised_learning//), the curse of dimensionality causes the *distance* between two observations to lose significance, making it harder to distinguish observations or groups of observations (clusters) from each other.

Below, we will implement Principal Component Analysis (PCA) as a method for dimensionality reduction in Python. For this, we will again use the breast cancer dataset from the previous session.

In [ ]:
# load breast cancer data
from sklearn import datasets
breast_cancer = datasets.load_breast_cancer(as_frame=True)

# divide data into features and targets
x_breast_cancer = breast_cancer["data"]
y_breast_cancer = breast_cancer["target"]

x_breast_cancer.head()

First, we want to examine whether the features of observations with cancer systematically differ from those without cancer. To do this, we will visualize the data using histograms and scatter plots.

In [ ]:
import seaborn as sns
# since our dataset has 30 features (columns), we will restrict ourselves
# for illustration purposes to the first four features
interesting_columns = ["mean radius", "mean texture", "mean perimeter", "mean area"]

# to easily visualize the data with seaborn, we create a DataFrame
# containing the relevant features and the target values
plot_data_breast_cancer = x_breast_cancer[interesting_columns].copy()
plot_data_breast_cancer["target"] = y_breast_cancer.copy()

`seaborn` provides the convenient function `PairGrid()` (see [documentation](https://seaborn.pydata.org/generated/seaborn.PairGrid.html)). It creates a grid of plots with as many rows and columns as there are features in the dataset.

* On the **diagonal** of the grid, a histogram of the feature values is shown for each feature.
* **Off the diagonal**, a scatter plot is shown for each pair of features.

In total, this produces a scatter plot for every possible combination of two features.

In [ ]:
# create a grid of plots
g = sns.PairGrid(plot_data_breast_cancer, hue="target")
# specify that histograms should appear on the diagonal
g.map_diag(sns.histplot)
# specify that scatter plots should appear off the diagonal
g.map_offdiag(sns.scatterplot, alpha=0.5)
# manually add a legend, since it is not generated automatically
g.add_legend();

We can see that the scatter plots for observations with cancer and those without do differ—but there is also a relatively large overlap between the regions. Can we separate the scatter plots more clearly?

Principal Component Analysis (PCA) can help with this. As illustrated below, the idea of PCA is to *project* the original dimensions of the dataset onto new dimensions that make it easier to distinguish the scatter plots from each other.

In [ ]:
from IPython.display import Image
Image(url='https://upload.wikimedia.org/wikipedia/commons/9/9c/PCA_Projection_Illustration.gif')

In the [lecture](https://janalasser.at/lectures/MD_KI/VO3_3_data_transformations/#/2/3/2), we already learned about PCA as a tool for dimensionality reduction. `scikit-learn` also provides functions for PCA, which work in ways very similar to those we have already used for supervised learning.

In [ ]:
# import the PCA "model" from scikit-learn
from sklearn.decomposition import PCA

# instantiate an empty PCA model and set the hyperparameters
# since we only want to keep the first two principal components of the data,
# set n_components=2
pca_breast_cancer = PCA(n_components=2)

# finally, fit the model on the breast cancer data
pca_breast_cancer.fit(x_breast_cancer)

From this point, the use of `pca()` differs from supervised learning: we do not want to predict values for individual observations, but rather *transform* the entire dataset. Therefore, we use the `transform()` function of the trained model.

In [ ]:
# transform the data by projecting it onto the first two principal components
# the resulting dataset now has only 2 dimensions instead of 30
x_breast_cancer_pca = pca_breast_cancer.transform(x_breast_cancer)
print("Original dimensions of the dataset: {}".format(str(x_breast_cancer.shape)))
print("Reduced dimensions of the dataset: {}".format(str(x_breast_cancer_pca.shape)))

In [ ]:
# unfortunately, the column names are lost during the transformation process
x_breast_cancer_pca

In [ ]:
# therefore, we create a pandas DataFrame from the transformed
# data and manually add column names
import pandas as pd
x_breast_cancer_pca = pd.DataFrame(data=x_breast_cancer_pca, columns=["PC1", "PC2"])
x_breast_cancer_pca.head()

In [ ]:
# to visualize the data, we also add the "target" column with the
# values from the original (non-transformed) data
x_breast_cancer_pca["target"] = y_breast_cancer.copy()

# visualize the data projected onto the first two principal components
# using a scatter plot
sns.scatterplot(data=x_breast_cancer_pca, x="PC1", y="PC2", hue="target", alpha=0.3)

The idea behind Principal Component Analysis is to capture as much of the dataset's *variance* as possible in a few principal components, thereby effectively reducing the dataset's dimensionality. How much variance is explained by each principal component? This information is contained in the `explained_variance_ratio_` attribute of the trained PCA model:

In [ ]:
pca_breast_cancer.explained_variance_ratio_

In [ ]:
# we can also visualize this with a bar chart.
# first, we create a suitable DataFrame for this purpose ...
explained_variance = pd.DataFrame()
explained_variance["component"] = range(1, len(pca_breast_cancer.explained_variance_ratio_) + 1)
explained_variance["explained variance"] = pca_breast_cancer.explained_variance_ratio_

# ... and then use seaborn's barplot() function
sns.barplot(data=explained_variance, x="component", y="explained variance")

Each principal component consists of a mixture (linear combination) of the original features in the dataset. We can also examine which features are particularly important for each principal component. This information is contained in the `components_` attribute of the trained PCA model. We can access the different principal components by their index: `components_[0]` contains the information about how important each feature is for the first principal component, `components_[1]` for the second, and so on.

In [ ]:
# we can also visualize the importance of features for the principal components
# by creating a pandas DataFrame for this purpose
feature_importance_breast_cancer = pd.DataFrame()
feature_importance_breast_cancer['feature'] = x_breast_cancer.columns
feature_importance_breast_cancer['PC1'] = pca_breast_cancer.components_[0]
feature_importance_breast_cancer['PC2'] = pca_breast_cancer.components_[1]

In [ ]:
# importance of features in the first principal component
sns.barplot(data=feature_importance_breast_cancer, y='feature', x='PC1')

In [ ]:
# importance of features in the second principal component
sns.barplot(data=feature_importance_breast_cancer, y='feature', x='PC2')

## Excursus: Scaling

So far, we have used the data "as is." However, even for PCA (and later for unsupervised learning or clustering), it can be useful to first transform the data to bring the features into a similar range.

Below, we will do this manually for the feature `mean radius` in the dataset:

In [ ]:
x_breast_cancer["mean radius"].head()

In [ ]:
# this is what the feature looks like before we transform it
sns.histplot(x_breast_cancer, x="mean radius")

In [ ]:
# as a first step, we "center" the feature by subtracting
# the mean of all observations
x_breast_cancer["mean radius centered"] = x_breast_cancer["mean radius"] - x_breast_cancer["mean radius"].mean()
x_breast_cancer["mean radius centered"].head()

In [ ]:
# this does not change the shape of the distribution, only its center
sns.histplot(x_breast_cancer, x="mean radius centered")

In [ ]:
# as a second step, we scale the data by dividing it by the
# standard deviation of all observations
x_breast_cancer["mean radius scaled"] = x_breast_cancer["mean radius centered"] / x_breast_cancer["mean radius"].std()
x_breast_cancer["mean radius scaled"].head()

In [ ]:
# again, this does not change the shape of the distribution,
# only the range of values it covers
sns.histplot(x_breast_cancer, x="mean radius scaled")

We do not have to transform the data "manually"—`scikit-learn` provides built-in functionality for this, too. The transformations we just performed (centering and scaling by the standard deviation) are automatically applied to all features (columns) in the dataset by `StandardScaler()`:

In [ ]:
# first, we remove the two columns that we manually added earlier,
# as we will no longer need them later
x_breast_cancer = x_breast_cancer.drop(columns=["mean radius centered", "mean radius scaled"])

In [ ]:
# import the StandardScaler from scikit-learn
from sklearn.preprocessing import StandardScaler

# we create an "empty" scaler ...
scaler_breast_cancer = StandardScaler()

# ... and "fit" it to the dataset. In the case of StandardScaler,
# the "fitting" step doesn’t actually change anything - it just passes
# the data to the function
scaler_breast_cancer.fit(x_breast_cancer)

# scale the data using the scaler’s transform() function
# the function returns the scaled data, which we store in a new variable
x_breast_cancer_scaled = scaler_breast_cancer.transform(x_breast_cancer)

# unfortunately, the column names are again lost, so we manually add them back
x_breast_cancer_scaled = pd.DataFrame(x_breast_cancer_scaled, columns=x_breast_cancer.columns)
x_breast_cancer_scaled.head()

Now we want to explore whether transforming the data helps the PCA better separate the two classes ("cancer" vs. "no cancer").

In [ ]:
# we instantiate a new PCA model and fit it with the scaled breast cancer data
pca_breast_cancer_scaled = PCA(n_components=2)
pca_breast_cancer_scaled.fit(x_breast_cancer_scaled)

# we project the scaled breast cancer data onto the first two principal components
x_breast_cancer_pca_scaled = pca_breast_cancer_scaled.transform(x_breast_cancer_scaled)

# the column names were lost here as well, so we add them back
# before we visualize the data
x_breast_cancer_pca_scaled = pd.DataFrame(data=x_breast_cancer_pca_scaled, columns=["PC1", "PC2"])
x_breast_cancer_pca_scaled["target"] = y_breast_cancer

# visualization with a scatter plot
sns.scatterplot(data=x_breast_cancer_pca_scaled, x="PC1", y="PC2", hue="target", alpha=0.5)

In [ ]:
# the explained variance is now distributed more evenly across the
# two principal components
explained_variance_scaled = pd.DataFrame()
explained_variance_scaled["component"] = range(1, len(pca_breast_cancer_scaled.explained_variance_ratio_) + 1)
explained_variance_scaled["explained variance"] = pca_breast_cancer_scaled.explained_variance_ratio_
sns.barplot(data=explained_variance_scaled, x="component", y="explained variance")

In [ ]:
# and more features contribute to the principal components
feature_importance_breast_cancer_scaled = pd.DataFrame()
feature_importance_breast_cancer_scaled['feature'] = x_breast_cancer_scaled.columns
feature_importance_breast_cancer_scaled['PC1'] = pca_breast_cancer_scaled.components_[0]
feature_importance_breast_cancer_scaled['PC2'] = pca_breast_cancer_scaled.components_[1]

sns.barplot(data=feature_importance_breast_cancer_scaled, y='feature', x='PC1')

In [ ]:
sns.barplot(data=feature_importance_breast_cancer_scaled, y='feature', x='PC2')

<font color='blue'>**Exercise 1**</font>  
<font color='blue'>In the code cell below, a dataset containing measurements of flower features belonging to different species is loaded. If you're curious, you can read more about this dataset [here](https://en.wikipedia.org/wiki/Iris_flower_data_set).
<ul class="outside">
<li><font color='blue'>Familiarize yourself with the dataset: how many observations are there? How many features, and what do they represent?</font></li>
<li><font color='blue'>Scale the data by subtracting the mean and dividing by the standard deviation for each feature. Hint: use `StandardScaler()` from `scikit-learn`!</font></li>
<li><font color='blue'>Perform a Principal Component Analysis (PCA) with `n_components=2`. How much variance is captured by the first principal component (PC 1), and how much by the second (PC 2)?</font></li>
<li><font color='blue'>Which features of the dataset are most important for the first principal component (PC 1), and which for the second (PC 2)?</font></li>
</ul>  
<font color='blue'></font>

In [ ]:
iris = datasets.load_iris(as_frame=True)
x_iris = iris["data"]
y_iris = iris["target"]

plot_data_iris = x_iris.copy()
plot_data_iris["target"] = y_iris.copy()

g = sns.PairGrid(plot_data_iris, hue="target")
g.map_diag(sns.histplot)
g.map_offdiag(sns.scatterplot, alpha=0.5)
g.add_legend();

In [ ]:
# Scaling: Your code here

In [ ]:
# PCA: Your code here

In [ ]:
# Explained varianec: Your code here

In [ ]:
# Feature importance: Your code here

## Clustering with K-Means

So far, for illustration purposes, we've always used the **class labels** — like "cancer" vs. "no cancer" for the breast cancer dataset, or "setosa," "versicolor," and "virginica" for the flowers. But what if we *don't* know the class labels and want to *discover* them instead? That's the classic use case for **unsupervised learning**, namely [cluster analysis](https://janalasser.at/lectures/MD_KI/VO3_1_unsupervised_learning/#/1/0/2).

To demonstrate the concept, we'll use a *real-world* dataset containing various **features of farms**, which we also briefly discussed in the [lecture](https://janalasser.at/lectures/MD_KI/VO3_1_unsupervised_learning/#/1/2/2). There's even a research project behind it — if you're curious, you can find details in this [publication](https://www.nature.com/articles/s41598-021-00469-2). Our goal will be to find out whether farms can be **grouped into distinct clusters** based on their characteristics.

In [ ]:
# we load the dataset containing farm characteristics into a pandas DataFrame
# it has 160 features but no target labels
url = "https://drive.google.com/uc?id=1mNO7yf89ReYPvjJgfD3YdY5MdIVGvCtp"
farm = pd.read_csv(url)
farm.head()

In [ ]:
# as with the breast cancer data, we scale the dataset...
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(farm)
farm_scaled = scaler.transform(farm)
farm_scaled = pd.DataFrame(farm_scaled, columns=farm.columns)

In [ ]:
# ... and conduct a PCA for dimensionality reduction
from sklearn.decomposition import PCA
pca_farm = PCA(n_components=2)
pca_farm.fit(farm_scaled)
farm_pca = pca_farm.transform(farm_scaled)
farm_pca = pd.DataFrame(data=farm_pca, columns=["PC1", "PC2"])
sns.scatterplot(data=farm_pca, x="PC1", y="PC2", alpha=0.3)

Now, as described in the [lecture](https://janalasser.at/lectures/MD_KI/VO3_1_unsupervised_learning/#/0/2/0), we want to perform a cluster analysis to identify different clusters (which we can then interpret as "classes"). In the [lecture](https://janalasser.at/lectures/MD_KI/VO3_4_algorithms_unsupervised_learning/#/4), we learned about the K-Means algorithm for this purpose. It finds clusters by starting from an initial (random) clustering and iteratively moving observations between clusters until the assignment of observations to clusters is optimal.

In [ ]:
from IPython.display import Image
Image(url='https://upload.wikimedia.org/wikipedia/commons/e/ea/K-means_convergence.gif')

`scikit-learn` also provides a ready-made implementation of the K-Means algorithm, which we can use following the usual workflow. K-Means also has hyperparameters:

* `n_clusters`: we must specify in advance how many clusters we expect. For the farm dataset, we don’t really know the optimal number of clusters, so we can either guess or systematically try different values.
* `random_state`: since clustering involves a random component (i.e., which cluster each observation is initially assigned to), it is good practice to fix the random state to make the results reproducible.

In [ ]:
# import the k-means model
from sklearn.cluster import KMeans

# instantiate an empty model and set the hyperparameters
# for the first attempt, we choose three predefined clusters
kmeans_farm = KMeans(n_clusters=3, random_state=42)

# fit the algorithm on the first two principal components of the
# farm data
kmeans_farm.fit(farm_pca)

# predict the cluster assignment of the observations
cluster_farm = kmeans_farm.predict(farm_pca)

In [ ]:
cluster_farm

In [ ]:
# to visualize cluster membership, we create a DataFrame
# containing the principal components and the predicted cluster assignments
plot_data_farm = farm_pca.copy()
plot_data_farm["cluster"] = cluster_farm

In [ ]:
sns.scatterplot(data=plot_data_farm, x="PC1", y="PC2", hue="cluster", alpha=0.5, palette=sns.color_palette("tab10"))

As discussed in the [lecture](https://janalasser.at/lectures/MD_KI/VO3_4_algorithms_unsupervised_learning/#/4/3/1), the outcome of the K-Means algorithm depends on the initial clustering. This initial assignment is chosen randomly — by setting `random_state` to a fixed number, we ensure it is always the same. What happens if we choose a different `random_state` and thus a different initial clustering?

In [ ]:
# we set the random_state to 11 (previously 42) and run the algorithm again
kmeans_farm = KMeans(n_clusters=3, random_state=11)
kmeans_farm.fit(farm_pca)

# predict the cluster assignment of the observations
cluster_farm_new_random_state = kmeans_farm.predict(farm_pca)

# visualize the data with four clusters
plot_data_farm["cluster_4_rnd"] = cluster_farm_new_random_state
sns.scatterplot(data=plot_data_farm, x="PC1", y="PC2", hue="cluster_4_rnd", alpha=0.5, palette=sns.color_palette("tab10"));

What happens if we set the number of clusters to `n_clusters=4`?

In [ ]:
# instantiate an empty model and set the hyperparameters
kmeans_farm = KMeans(n_clusters=4, random_state=11)

# fit the algorithm on the first two principal components of the
# farm data
kmeans_farm.fit(farm_pca)

# predict the cluster assignment of the observations
cluster_farm_4 = kmeans_farm.predict(farm_pca)

In [ ]:
cluster_farm_4

In [ ]:
# visualize the data with four clusters
plot_data_farm["cluster_4"] = cluster_farm_4
sns.scatterplot(data=plot_data_farm, x="PC1", y="PC2", hue="cluster_4", alpha=0.5, palette=sns.color_palette("tab10"));

## Finding the Optimal Number of Clusters



How can we find the best number of clusters? We can try to measure the quality of the clustering. For this, we need a metric to assess clustering quality.

In the [lecture](https://janalasser.at/lectures/MD_KI/VO3_4_algorithms_unsupervised_learning/#/4/2/4), we discussed the silhouette coefficient as a way to measure clustering quality. It measures how similar an observation is to other observations in the same cluster (cohesion) compared to observations in other clusters (separation).

![silhouette score](https://drive.google.com/uc?id=17x-xSwIN5QUaB_IJ_Uw8IAgxAz7Pv0eF)

`scikit-learn` also provides a function `silhouette_score()` in the `metrics` module that can be easily used (see also [documentation](https://scikit-learn.org/1.5/modules/generated/sklearn.metrics.silhouette_score.html)).

Rule of thumb: a clustering with a silhouette coefficient >= 0.7 is considered a "strong" clustering, a coefficient >= 0.5 is "satisfactory," and a coefficient >= 0.25 is "weak." Values below this suggest there is no clear cluster structure in the data. Special caution is needed for high-dimensional data because the [curse of dimensionality](https://janalasser.at/lectures/MD_KI/VO3_3_data_transformations/#/2/2/0) generally lowers the silhouette values.

Alternatively, we can use a clustering algorithm that automatically finds the optimal number of clusters, such as DBSCAN (see exercise).

In [ ]:
# we measure the silhouette coefficient of the clustering of the farms
# with 3 and 4 predefined clusters:
from sklearn.metrics import silhouette_score
sil_score_3 = silhouette_score(farm_pca, cluster_farm)
sil_score_4 = silhouette_score(farm_pca, cluster_farm_4)

print(f"Silhouette score with n_clusters=3: {sil_score_3}")
print(f"Silhouette score with n_clusters=4: {sil_score_4}")

<font color='blue'>**Exercise 2**</font>  
<ul class="outside">
<li><font color='blue'>Führen Sie das K Means Clustering für den "Iris"-Datensatz durch. Wählen Sie dafür einmal <tt>n_clusters=3</tt> und einmal <tt>n_clusters=2</tt>. Setzen Sie für beide Fälle den random state auf <tt>random_state=42</tt>. Speichern Sie die Ergebnisse jeweils in separaten Variablen.</font></li>
<li><font color='blue'>Visualisieren Sie das Clustering für beide Fälle mit Streudiagrammen wie schon für den Bauernhof-Datensatz.</font></li>
<li><font color='blue'>Berechnen Sie den Silhouettenkoeffizient für beide Fälle. Für welche Clusteranzahl ist er besser?</font></li>

In [ ]:
# Your code here

## Homework

The homework for this course section can be found in [this notebook](https://colab.research.google.com/drive/15a43ZANMgSNSIbAW8UrZ0354E8sPMsks?usp=sharing).

## Additional Materials
* **machine learning**: [Book](https://www.amazon.de/Introduction-Machine-Learning-Python-Scientists/dp/1449369413?shipTo=AT&source=ps-sl-shoppingads-lpcontext&ref_=fplfs&psc=1&smid=A3JWKAKR8XB7XF&language=de_DE&gQT=1) *Introduction to Machine Learning with Python: A Guide for Data Scientists* with extensive [examples and exercises](https://github.com/amueller/introduction_to_ml_with_python) in Python.

## Source and License

This notebook was created by Jana Lasser for Course B "Technical Aspects" of the Microcredential "AI and Society" at the University of Graz.

The notebook can be used, modified, and redistributed under the terms of the [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0) license.

This notebook was translated from German using GPT-5 and cross-checked by Alina Herderich.